_Spørgsmål: Er min betingelse for udvælgning af current bygninger i overenestemmelse med datafordelerens_

fremgangsmåde: Jeg henter både den bitemporale og current udgave af BBRbygning fra datafordeleren. Herefter bruger jeg MIN current betingelse på det bitemporale data. Forhåbningen er da, at det filtrede datasæt matcher med current datasættet fra datafordeleren.

In [1]:
# indlæser pakker
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

In [ ]:
# første betinkelighed: er virkningFra og virkningTil retvisende?
# og gør det sig gælende for hele det relevante tidsinterval (lokalebasen data strækker sig fra 2009-2026)
path_current            = r"D:\tung_databehandling\BBR\raw\til_temporal_filtrering_tjek\BBR_V3_Bygning_TotalDownload_csv_Current_653.csv"

path_bitemp             = r"D:\tung_databehandling\BBR\raw\til_temporal_filtrering_tjek\BBR_V3_Bygning_TotalDownload_csv_Bitemporal_653.csv"
path_bitemp_prefiltered = r"D:\tung_databehandling\BBR\raw\til_temporal_filtrering_tjek\BBR_V3_Bygning_TotalDownload_csv_Bitemporal_653_filtered.csv"
# AAD: Active At Date
path_bitemp_AAD         = r"D:\tung_databehandling\BBR\raw\til_temporal_filtrering_tjek\BBR_V3_Bygning_TotalDownload_csv_Bitemporal_653_filtered_AAD.csv"

In [4]:
def prefilter_BBRbygning(df_path, path_output, chunksize=1_000_000, relevant=True, status=None):

    n_start = len(df_path)

    first_write = True
    n_slut = 0
    count_ikkerelevant = 0
    count_ikkestatus   = 0
    for chunk in pd.read_csv(df_path, chunksize=chunksize):
        
        print(".", end="")

        # betingelse: relevant
        if relevant:
            mask_relevant = chunk['registreringTil'].isna()
            count_ikkerelevant = sum(~mask_relevant)
            chunk = chunk[mask_relevant]

        
        # betingelse: status er i inputtet "status"
        if status is not None:
            if not isinstance(status, (list, tuple, set)):
                status = [status]
            mask_status = (chunk["status"].isin(status))
            count_ikkestatus = sum(~mask_status)
            print(count_ikkestatus)
            chunk[mask_status]

        # fjern forkerte "Z"'ere
        for dim in ['byg406Koordinatsystem', 'registreringFra', 'virkningFra', 'virkningTil']:
            if dim in chunk.columns:
                chunk[dim] = chunk[dim].astype(str).str.replace("Z","")

        n_slut += len(chunk)

        chunk.to_csv(
            f"{path_output}",
            mode="w" if first_write else "a",
            index=False,
            header=first_write
        )
        first_write = False

    print("\nFiltrering af BBR-bygninger afsluttet")
    print("-" * 40)
    print(f"Antal rækker i inputfilen:      {n_start:,}")
    
    if relevant:
        print(f"Fjernet som ikke relevante:     {count_ikkerelevant:,}")

    if status is not None:
        print(f"Fjernet pga. status:             {count_ikkestatus:,}")
        print(f"Valgte statusværdier:            {list(status)}")

    print(f"Antal rækker i outputfilen:     {n_slut:,}")
    print(f"Antal rækker fjernet i alt:     {n_start - n_slut:,}")
    print(f"Output gemt i:                   {path_output}")
    

In [5]:
# filterer så det kun er registreringTil.isna() og status = <valg> (fx status = 6)
prefilter_BBRbygning(path_bitemp, path_bitemp_AAD, status=6)

.80599
.73947
.75220
.74858
.73905
.73705
.74330
.73298
.69313
.23820
.29649
.28993
.31981
.46655
.37718
.109673
.45956
.109268
.141403
.135978
.114560
.194934
.153072
.169420
.164368
.150575
.166083
.182892
.172675
.126536
.73385
.73667
.73345
.73928
.73531
.72917
.73081
.71749
.72229
.71647
.71393
.71429
.78048
.143785
.200697

Filtrering af BBR-bygninger afsluttet
----------------------------------------
Antal rækker i inputfilen:      111
Fjernet som ikke relevante:     307,483
Fjernet pga. status:             200,697
Valgte statusværdier:            [6]
Antal rækker i outputfilen:     33,748,531
Antal rækker fjernet i alt:     -33,748,420
Output gemt i:                   D:\tung_databehandling\BBR\raw\til_temporal_filtrering_tjek\BBR_V3_Bygning_TotalDownload_csv_Bitemporal_653_filtered_AAD.csv


In [6]:
def iscurrent_filter(path_input, path_output, yyyyMMdd, virkningcols = ["virkningFra", "virkningTil"], chunksize=1_000_000):

    # virkningFra og virkningTil parses til pd.datetime
    virkningcols = np.array(virkningcols)

    # yyyyMMdd parses til pd.datetime
    try:
        yyyyMMdd = pd.Timestamp(yyyyMMdd, tz="UTC")
    except:
        raise ValueError(f"den valgte current-dato (dato du vil have BBR for): {yyyyMMdd} kan ikke konverteres til pd.Timestamp")

    first_write = True
    n_total = 0
    n_current = 0
    for i, chunk in enumerate(pd.read_csv(path_input, chunksize=chunksize), start=1):

        if np.isin(virkningcols, chunk.columns).all() != 1:
            raise ValueError(f"valgte virkningscols: {virkningcols} findes ikke i datasættet, som har følgende kolonner: {chunk.columns}")

        chunk[virkningcols[0]] = pd.to_datetime(chunk[virkningcols[0]], errors="coerce", utc=True)
        chunk[virkningcols[1]] = pd.to_datetime(chunk[virkningcols[1]], errors="coerce", utc=True)

        # betingelse for at være en current bygning
        mask_existed_at_date      = chunk[virkningcols[0]] <= yyyyMMdd
        mask_still_exists_at_date = (chunk[virkningcols[1]] > yyyyMMdd) | (chunk[virkningcols[1]].isna())
        mask_current_at_date      = mask_existed_at_date & mask_still_exists_at_date

        chunk_current = chunk[mask_current_at_date]

        n_total += len(chunk)
        n_current += len(chunk_current)

        print(
            f"chunk {i}: {len(chunk_current):,} ud af "
            f"{len(chunk):,} rækker er current "
            f"({len(chunk_current) / len(chunk):.2%})"
        )

        if len(chunk_current) > 0:
            chunk_current.to_csv(
                f"{path_output}",
                mode="w" if first_write else "a",
                index=False,
                header=first_write
            )
            first_write = False

    print(f"\nskrev {n_current:,} ud af {n_total:,} rækker til {path_output}")
    print(f"samlet andel current: {n_current / n_total:.2%}")

In [7]:
# filterer data med registreringTIl.isna og status = 6, efter datoen som current filen blev downloaded.
# altså hvis mit filter fungerer som det burde, da bør denne filterering udvælge præcis det samme som current datasættet.
iscurrent_filter(path_bitemp_prefiltered, path_bitemp_AAD, "2026-09-01")

chunk 1: 75,156 ud af 1,000,000 rækker er current (7.52%)
chunk 2: 67,913 ud af 1,000,000 rækker er current (6.79%)
chunk 3: 67,331 ud af 1,000,000 rækker er current (6.73%)
chunk 4: 67,407 ud af 1,000,000 rækker er current (6.74%)
chunk 5: 67,509 ud af 1,000,000 rækker er current (6.75%)
chunk 6: 67,726 ud af 1,000,000 rækker er current (6.77%)
chunk 7: 68,572 ud af 1,000,000 rækker er current (6.86%)
chunk 8: 274,581 ud af 1,000,000 rækker er current (27.46%)
chunk 9: 285,350 ud af 1,000,000 rækker er current (28.54%)
chunk 10: 276,422 ud af 1,000,000 rækker er current (27.64%)
chunk 11: 285,501 ud af 1,000,000 rækker er current (28.55%)
chunk 12: 281,501 ud af 1,000,000 rækker er current (28.15%)
chunk 13: 299,786 ud af 1,000,000 rækker er current (29.98%)
chunk 14: 291,742 ud af 1,000,000 rækker er current (29.17%)
chunk 15: 283,934 ud af 1,000,000 rækker er current (28.39%)
chunk 16: 320,356 ud af 1,000,000 rækker er current (32.04%)
chunk 17: 324,047 ud af 1,000,000 rækker er cur

In [ ]:
# load datasæt (current fra datafordeleren og bitemp_current fra min temporale-betingelse)
# vi vil gerne have at disse er en (understøtter at sorteringen er korrekt)
nrows = None
df_bitemp_current = pd.read_csv(path_bitemp_AAD, nrows=nrows)
df_current        = pd.read_csv(path_current ,nrows=nrows)

display(df_bitemp_current.head())
display(df_current.head())

print(df_bitemp_current["status"].value_counts())
print(df_current["status"].value_counts())

,datafordelerOpdateringstid,datafordelerRowId,datafordelerRowVersion,datafordelerRegisterImportSequenceNumber,forretningshændelse,forretningsområde,forretningsproces,id_namespace,id_lokalId,kommunekode,...,byg500Notatlinjer,byg150Gulvbelægning,byg151Frihøjde,byg152ÅbenLukketKonstruktion,byg153Konstruktionsforhold,byg140ServitutForUdlejningsEjendomDato,jordstykke,husnummer,ejerlejlighed,grund
0,2021-04-04T17:27:45.602979Z,a778ae5e-e8f1-41b0-bdbd-290aaf8f82d0,1,1857504,Bygning,54.15.05.05,0.0,http://data.gov.dk/bbr/bygning,4db2e5db-b83c-4531-bfbb-9c693a287aa4,101,...,NaN,NaN,NaN,NaN,NaN,NaN,10410750.0,0a3f507a-6448-32b8-e044-0003ba298018,NaN,e1a35f74-2dc7-419f-bbf2-19f5d9ea9c89
1,2021-04-04T17:27:45.604473Z,55d9fc9e-375f-4dc9-bc9a-7f134541f56b,1,1857504,Bygning,54.15.05.05,3.0,http://data.gov.dk/bbr/bygning,9fa1f376-37dc-4f01-b5d9-9c693bec4e5a,260,...,NaN,NaN,NaN,NaN,NaN,NaN,1674987.0,0a3f507e-f1f1-32b8-e044-0003ba298018,NaN,ea0553a4-60b8-4397-aa79-63b7f8ecfad2
2,2021-04-04T17:27:45.701620Z,c46cf2b0-6565-4d54-a753-ea441d215e85,1,1857504,Bygning,54.15.05.05,0.0,http://data.gov.dk/bbr/bygning,dcee8cfa-0472-434c-861e-9c694d6dea7b,760,...,NaN,NaN,NaN,NaN,NaN,NaN,621806.0,0a3f5093-0061-32b8-e044-0003ba298018,NaN,424cd3d2-6782-40bd-90b9-ea376de653fd
3,2021-04-04T17:27:45.708630Z,eb312c39-308f-4d74-8985-4a2b2e216d9a,1,1857504,Bygning,54.15.05.05,2.0,http://data.gov.dk/bbr/bygning,15ee4527-8e18-4625-be36-9c69513a062b,849,...,NaN,NaN,NaN,NaN,NaN,NaN,2000717.0,0a3f5099-abf6-32b8-e044-0003ba298018,NaN,d87f1401-3d30-47d0-9ed1-063f3e9aa786
4,2021-04-04T17:27:45.826837Z,7c15105e-fcff-4b9a-bcad-a397cdf1da42,1,1857504,Bygning,54.15.05.05,4.0,http://data.gov.dk/bbr/bygning,a688d576-d71f-43a5-bc58-9c6968716ebb,615,...,NaN,NaN,NaN,NaN,NaN,NaN,1116168.0,0a3f508f-9763-32b8-e044-0003ba298018,NaN,9a496c72-c28b-474b-98dd-d475ff284ed5


,datafordelerOpdateringstid,datafordelerRowId,datafordelerRowVersion,datafordelerRegisterImportSequenceNumber,forretningshændelse,forretningsområde,forretningsproces,id_namespace,id_lokalId,kommunekode,...,byg500Notatlinjer,byg150Gulvbelægning,byg151Frihøjde,byg152ÅbenLukketKonstruktion,byg153Konstruktionsforhold,byg140ServitutForUdlejningsEjendomDato,jordstykke,husnummer,ejerlejlighed,grund
0,2021-04-04T17:27:45.602979Z,a778ae5e-e8f1-41b0-bdbd-290aaf8f82d0,1,1857504,Bygning,54.15.05.05,0.0,http://data.gov.dk/bbr/bygning,4db2e5db-b83c-4531-bfbb-9c693a287aa4,101,...,NaN,NaN,NaN,NaN,NaN,NaN,10410750.0,0a3f507a-6448-32b8-e044-0003ba298018,NaN,e1a35f74-2dc7-419f-bbf2-19f5d9ea9c89
1,2021-04-04T17:27:45.604473Z,55d9fc9e-375f-4dc9-bc9a-7f134541f56b,1,1857504,Bygning,54.15.05.05,3.0,http://data.gov.dk/bbr/bygning,9fa1f376-37dc-4f01-b5d9-9c693bec4e5a,260,...,NaN,NaN,NaN,NaN,NaN,NaN,1674987.0,0a3f507e-f1f1-32b8-e044-0003ba298018,NaN,ea0553a4-60b8-4397-aa79-63b7f8ecfad2
2,2021-04-04T17:27:45.701620Z,c46cf2b0-6565-4d54-a753-ea441d215e85,1,1857504,Bygning,54.15.05.05,0.0,http://data.gov.dk/bbr/bygning,dcee8cfa-0472-434c-861e-9c694d6dea7b,760,...,NaN,NaN,NaN,NaN,NaN,NaN,621806.0,0a3f5093-0061-32b8-e044-0003ba298018,NaN,424cd3d2-6782-40bd-90b9-ea376de653fd
3,2021-04-04T17:27:45.708630Z,eb312c39-308f-4d74-8985-4a2b2e216d9a,1,1857504,Bygning,54.15.05.05,2.0,http://data.gov.dk/bbr/bygning,15ee4527-8e18-4625-be36-9c69513a062b,849,...,NaN,NaN,NaN,NaN,NaN,NaN,2000717.0,0a3f5099-abf6-32b8-e044-0003ba298018,NaN,d87f1401-3d30-47d0-9ed1-063f3e9aa786
4,2021-04-04T17:27:45.826837Z,7c15105e-fcff-4b9a-bcad-a397cdf1da42,1,1857504,Bygning,54.15.05.05,4.0,http://data.gov.dk/bbr/bygning,a688d576-d71f-43a5-bc58-9c6968716ebb,615,...,NaN,NaN,NaN,NaN,NaN,NaN,1116168.0,0a3f508f-9763-32b8-e044-0003ba298018,NaN,9a496c72-c28b-474b-98dd-d475ff284ed5


status
6     4834776
9      635829
10     442254
11     130000
3      117739
14      36784
2       32084
12       6662
13       1824
Name: count, dtype: int64
status
6     4834776
9      635829
10     442254
11     130000
3      117739
14      36784
2       32084
12       6662
13       1824
Name: count, dtype: int64
